# Mini-GPT en PyTorch — Baseline serio

Un Transformer decoder desde cero en PyTorch, entrenado sobre **Tiny Shakespeare**
(~1 MB de texto, se descarga solo). El notebook está pensado para ejecutarse de
principio a fin en **Google Colab Free (CPU o GPU T4)** sin que tengas que tocar nada.

**Lo que vas a tener al final:**
- Un mini-GPT (~1.5M parámetros) entrenado.
- Funciones de generación con `temperature`, `top_k` y `top_p`.
- Checkpoints guardados en `ckpt/`.
- Una base modular lista para iterar: cambiar dataset, ampliar tamaño, migrar a diálogo.

**Lo que NO es:** este modelo genera **texto coherente estilo Shakespeare**, no
conversaciones tipo chatbot. En la última sección explico cómo migrar a un dataset de
diálogo (DailyDialog, OpenSubtitles) si quieres que mantenga una conversación.

> Inspirado en [nanoGPT](https://github.com/karpathy/nanoGPT) de Andrej Karpathy,
> simplificado y comentado en español.


In [ ]:
# ============================================================
# 0. Setup — imports, seed, detección de GPU
# ============================================================
import os
import math
import time
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device detectado: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("⚡ Entrenamiento será rápido (~5-10 min).")
else:
    print("⚠️  Sin GPU: el entrenamiento funcionará pero más lento (~30-45 min).")
    print("   En Colab: Runtime → Change runtime type → T4 GPU.")

# Carpeta para checkpoints
CKPT_DIR = Path("ckpt")
CKPT_DIR.mkdir(exist_ok=True)
print(f"Checkpoints en: {CKPT_DIR.resolve()}")


## 1. Dataset — Tiny Shakespeare

Descargamos el archivo `input.txt` (~1 MB) y lo cargamos en memoria.
Tokenización a nivel de **carácter** (más simple y robusta que BPE para un baseline).


In [ ]:
# ============================================================
# 1. Dataset — descarga + tokenización char-level
# ============================================================
DATA_URL = "https://raw.githubusercontent.com/karpathy/nanoGPT/master/data/shakespeare_char/input.txt"
DATA_PATH = Path("input.txt")

if not DATA_PATH.exists():
    print(f"Descargando Tiny Shakespeare desde {DATA_URL} ...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print(f"OK: {DATA_PATH} ({DATA_PATH.stat().st_size} bytes)")
else:
    print(f"Ya existe: {DATA_PATH} ({DATA_PATH.stat().st_size} bytes)")

# Leer
with DATA_PATH.open("r", encoding="utf-8") as f:
    text = f.read()

print(f"Longitud del texto: {len(text):,} caracteres")

# Vocabulario char-level
chars = sorted(set(text))
VOCAB_SIZE = len(chars)
print(f"Tamaño del vocabulario (char-level): {VOCAB_SIZE}")

# Tokenizador: char <-> int
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s: str) -> list[int]:
    return [stoi[c] for c in s if c in stoi]

def decode(ids: list[int]) -> str:
    return "".join(itos[i] for i in ids)

# Codificar todo el corpus a tensores
data = torch.tensor(encode(text), dtype=torch.long, device="cpu")
print(f"Tensor de datos: shape={data.shape}, dtype={data.dtype}")

# Split train/val 90/10
n_split = int(0.9 * len(data))
train_data = data[:n_split]
val_data = data[n_split:]
print(f"Train: {len(train_data):,} tokens | Val: {len(val_data):,} tokens")

# Ejemplo
print("\n--- Ejemplo ---")
sample = decode(encode("To be, or not to be:"))
print(f"Texto: {sample!r}")
print(f"IDs:   {encode(sample)}")


## 2. Configuración del modelo y data

Hiperparámetros conservadores para que el modelo entrene rápido en Colab free
y produzca texto coherente. Total ~1.5M parámetros.


In [ ]:
# ============================================================
# 2. Configuración
# ============================================================
@dataclass
class GPTConfig:
    block_size: int = 128       # contexto máximo (tokens)
    vocab_size: int = VOCAB_SIZE
    n_layer: int = 4            # número de bloques Transformer
    n_head: int = 4             # cabezas de atención
    n_embd: int = 128           # dimensión del embedding
    dropout: float = 0.1

# Hiperparámetros de entrenamiento
@dataclass
class TrainConfig:
    batch_size: int = 64
    epochs: int = 5
    lr: float = 1e-3
    weight_decay: float = 0.1
    warmup_iters: int = 100
    lr_decay_iters: int = 1000
    eval_interval: int = 200    # cada cuántos iters se evalúa en val
    eval_iters: int = 40        # iters usadas para estimar loss de val
    grad_clip: float = 1.0

cfg = GPTConfig()
tcfg = TrainConfig()

print("Config del modelo:", cfg)
print("Config de entrenamiento:", tcfg)


## 3. Modelo — Mini-GPT (Transformer decoder)

Implementación pura en PyTorch: `MultiHeadAttention`, `FeedForward`, `Block`,
`GPT`. Sin dependencias externas (no usamos `transformers` ni `tiktoken`).


In [ ]:
# ============================================================
# 3. Modelo Mini-GPT
# ============================================================
class MultiHeadSelfAttention(nn.Module):
    """Atención multi-cabeza causal (máscara triangular)."""
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        # buffer con máscara causal (no es parámetro, no se mueve con .to())
        self.register_buffer("mask",
                             torch.tril(torch.ones(block_size, block_size))
                                  .view(1, 1, block_size, block_size),
                             persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        # qkv: (B, T, 3C) -> (B, T, 3, n_head, head_dim) -> (3, B, n_head, T, head_dim)
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # cada uno (B, n_head, T, head_dim)

        # Atención escalada
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        out = att @ v  # (B, n_head, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class FeedForward(nn.Module):
    """MLP de dos capas con GELU."""
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd, bias=False),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Bloque Transformer: LayerNorm → Attn → residual → LayerNorm → FFN → residual."""
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = MultiHeadSelfAttention(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.ff = FeedForward(cfg.n_embd, cfg.dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    """Transformer decoder."""
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        # Inicialización (recomendado por Karpathy)
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                torch.nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"Contexto {T} > block_size {self.cfg.block_size}"

        tok = self.tok_emb(idx)            # (B, T, C)
        pos = self.pos_emb(torch.arange(T, device=idx.device))  # (T, C)
        x = self.drop(tok + pos)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)              # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        """Genera texto autoregresivamente."""
        self.eval()
        for _ in range(max_new_tokens):
            # Recortar contexto a block_size
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _ = self.forward(idx_cond)
            logits = logits[:, -1, :] / temperature  # (B, vocab)

            # Top-k
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            # Top-p (nucleus)
            if top_p is not None:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                probs = F.softmax(sorted_logits, dim=-1)
                cum_probs = torch.cumsum(probs, dim=-1)
                sorted_mask = cum_probs > top_p
                # Shift right para mantener al menos un token
                sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
                sorted_mask[..., 0] = False
                indices_to_remove = sorted_mask.scatter(1, sorted_idx, sorted_mask)
                logits = logits.masked_fill(indices_to_remove, float('-inf'))

            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat([idx, next_tok], dim=1)
        return idx


# Instanciar y contar parámetros
model = MiniGPT(cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo creado en {DEVICE}")
print(f"Parámetros totales: {n_params:,} (~{n_params/1e6:.2f}M)")

# Sanity check: forward con un minibatch
xb = train_data[:cfg.block_size].unsqueeze(0).to(DEVICE)
yb = train_data[1:cfg.block_size+1].unsqueeze(0).to(DEVICE)
logits, loss = model(xb, yb)
print(f"Forward OK: logits shape = {logits.shape}, loss = {loss.item():.4f}")
print(f"Loss esperada (uniforme): {-math.log(1.0/VOCAB_SIZE):.4f}")


## 4. Preparación de datos para entrenamiento

Usamos un `Dataset` que genera ventanas deslizantes de tamaño `block_size+1`.
El target es el mismo texto desplazado un token (predicción del siguiente token).


In [ ]:
# ============================================================
# 4. Dataset y DataLoader
# ============================================================
class CharDataset(Dataset):
    """Ventanas aleatorias de longitud block_size+1."""
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return max(0, len(self.data) - self.block_size - 1)
    def __getitem__(self, idx):
        chunk = self.data[idx:idx + self.block_size + 1]
        x = chunk[:-1].clone()
        y = chunk[1:].clone()
        return x, y

train_ds = CharDataset(train_data, cfg.block_size)
val_ds = CharDataset(val_data, cfg.block_size)
print(f"Train dataset: {len(train_ds):,} muestras")
print(f"Val dataset:   {len(val_ds):,} muestras")

train_loader = DataLoader(train_ds, batch_size=tcfg.batch_size,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=tcfg.batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)

# Sanity check
xb, yb = next(iter(train_loader))
xb, yb = xb.to(DEVICE), yb.to(DEVICE)
print(f"Batch shape: x={xb.shape}, y={yb.shape}")
print(f"Ejemplo de decode: {decode(xb[0][:30].tolist())!r}")


## 5. Entrenamiento

Loop de entrenamiento con:
- AdamW con weight decay
- Warmup + cosine decay (lr scheduler manual)
- Gradient clipping
- Evaluación periódica en validación
- Guardado de checkpoint al final de cada epoch


In [ ]:
# ============================================================
# 5. Entrenamiento
# ============================================================
optim = torch.optim.AdamW(model.parameters(),
                          lr=tcfg.lr,
                          weight_decay=tcfg.weight_decay,
                          betas=(0.9, 0.99))

def get_lr(it):
    """Warmup lineal + decaimiento coseno."""
    if it < tcfg.warmup_iters:
        return tcfg.lr * (it + 1) / tcfg.warmup_iters
    if it >= tcfg.lr_decay_iters:
        return tcfg.lr * 0.1
    decay_ratio = (it - tcfg.warmup_iters) / (tcfg.lr_decay_iters - tcfg.warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return tcfg.lr * (0.1 + 0.9 * coeff)

@torch.no_grad()
def estimate_loss():
    """Estima loss en train y val (promedio de varios batches)."""
    model.eval()
    out = {}
    for name, loader in [('train', train_loader), ('val', val_loader)]:
        losses = []
        for i, (x, y) in enumerate(loader):
            if i >= tcfg.eval_iters:
                break
            x, y = x.to(DEVICE), y.to(DEVICE)
            _, loss = model(x, y)
            losses.append(loss.item())
        out[name] = sum(losses) / len(losses) if losses else float('nan')
    model.train()
    return out

# Loop principal
global_iter = 0
train_losses = []
val_losses = []
best_val = float('inf')

print(f"\n=== Entrenando en {DEVICE} ===")
model.train()
t_start = time.time()

for epoch in range(tcfg.epochs):
    epoch_t0 = time.time()
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        # LR scheduler
        lr = get_lr(global_iter)
        for pg in optim.param_groups:
            pg['lr'] = lr

        # Forward
        logits, loss = model(x, y)

        # Backward
        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip)
        optim.step()

        train_losses.append(loss.item())

        if global_iter % tcfg.eval_interval == 0:
            stats = estimate_loss()
            val_losses.append((global_iter, stats['val']))
            elapsed = time.time() - t_start
            print(f"epoch {epoch+1}/{tcfg.epochs} | iter {global_iter:5d} | "
                  f"lr {lr:.2e} | train_loss {stats['train']:.4f} | "
                  f"val_loss {stats['val']:.4f} | t {elapsed:.1f}s")
            if stats['val'] < best_val:
                best_val = stats['val']
                torch.save({
                    'model': model.state_dict(),
                    'config': cfg.__dict__,
                    'train_config': tcfg.__dict__,
                    'val_loss': stats['val'],
                    'iter': global_iter,
                }, CKPT_DIR / "best.pt")
        global_iter += 1

    # Checkpoint al final de epoch
    torch.save({
        'model': model.state_dict(),
        'config': cfg.__dict__,
        'train_config': tcfg.__dict__,
        'epoch': epoch + 1,
        'iter': global_iter,
    }, CKPT_DIR / f"epoch_{epoch+1}.pt")
    print(f"→ epoch {epoch+1} completada en {time.time()-epoch_t0:.1f}s | "
          f"best val_loss = {best_val:.4f}")

total = time.time() - t_start
print(f"\n✅ Entrenamiento completado en {total:.1f}s ({total/60:.2f} min)")
print(f"Mejor val_loss: {best_val:.4f}")


## 6. Visualizar curva de aprendizaje

Graficamos la loss de train (instantánea por iter) y val (suavizada).


In [ ]:
# ============================================================
# 6. Curvas de aprendizaje
# ============================================================
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.plot(train_losses, alpha=0.3, label='train (instant)')
if val_losses:
    its, vs = zip(*val_losses)
    ax.plot(its, vs, 'o-', label='val', markersize=4)
ax.set_xlabel('Iteración')
ax.set_ylabel('Loss')
ax.set_title('Curva de aprendizaje')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## 7. Generar texto

Función de inferencia con `temperature`, `top_k` y `top_p`.


In [ ]:
# ============================================================
# 7. Generación
# ============================================================
def generate_text(prompt: str, max_new_tokens: int = 200,
                  temperature: float = 0.8, top_k: int = 40, top_p: float = 0.9) -> str:
    """Genera texto a partir de un prompt."""
    model.eval()
    ctx_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    if ctx_ids.size(1) == 0:
        ctx_ids = torch.tensor([[0]], dtype=torch.long, device=DEVICE)
    # Recortar a block_size
    if ctx_ids.size(1) > cfg.block_size:
        ctx_ids = ctx_ids[:, -cfg.block_size:]
    out_ids = model.generate(
        ctx_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
    )
    return decode(out_ids[0].tolist())

# Cargar el mejor checkpoint
ckpt = torch.load(CKPT_DIR / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model'])
print(f"Checkpoint cargado (val_loss={ckpt['val_loss']:.4f}, iter={ckpt['iter']})\n")

# Probar varios prompts
for p in [
    "ROMEO:",
    "To be, or",
    "JULIET: What",
    "The king:",
]:
    print("=" * 60)
    print(f"PROMPT: {p!r}")
    print("-" * 60)
    print(generate_text(p, max_new_tokens=150, temperature=0.8, top_k=40, top_p=0.9))
    print()


In [ ]:
# ============================================================
# 8. Celda interactiva — genera tu propio prompt
# ============================================================
#@title Genera texto con tu propio prompt
PROMPT = "Once upon a time"  #@param {type:"string"}
MAX_TOKENS = 200            #@param {type:"integer"}
TEMPERATURE = 0.8           #@param {type:"slider", min:0.1, max:2.0, step:0.1}
TOP_K = 40                  #@param {type:"integer"}
TOP_P = 0.9                 #@param {type:"slider", min:0.1, max:1.0, step:0.05}

print(generate_text(PROMPT, max_new_tokens=MAX_TOKENS,
                    temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P))


## 9. Guardar y cargar checkpoint

Dos helpers para persistencia. El checkpoint incluye pesos + config.


In [ ]:
# ============================================================
# 9. Helpers de persistencia
# ============================================================
def save_model(path: str):
    torch.save({
        'model': model.state_dict(),
        'config': cfg.__dict__,
        'train_config': tcfg.__dict__,
    }, path)
    print(f"Guardado en {path}")

def load_model(path: str, device=DEVICE):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    new_cfg = GPTConfig(**ckpt['config'])
    new_model = MiniGPT(new_cfg).to(device)
    new_model.load_state_dict(ckpt['model'])
    return new_model, new_cfg

# Ejemplo (descomenta para usar):
# save_model("mini_gpt_final.pt")
# model, cfg = load_model("ckpt/best.pt")


## 10. Próximos pasos para **conversación real**

Este modelo genera **texto coherente estilo Shakespeare**. Para que mantenga una
conversación (turnos usuario/assistant con coherencia), tienes que cambiar dos cosas:

### 1. Cambiar el dataset
Reemplaza `Tiny Shakespeare` por un dataset de diálogo. Opciones descargables
automáticamente (vía HuggingFace `datasets`):

```python
# En Colab: primero instala
# !pip install datasets
from datasets import load_dataset

# Opción A — DailyDialog (diálogos cotidianos, ~14k turnos)
ds = load_dataset("daily_dialog", split="train")

# Opción B — OpenSubtitles (enorme, mucho más rico)
ds = load_dataset("opensubtitles", "en-es", split="train")
```

Formatea como texto:
```
<user>  Hello, how are you?
<assistant>  I'm fine, thanks. And you?
<user>  ...
```

### 2. Tokenización BPE (recomendado para diálogo)
La tokenización char-level está bien para Shakespeare, pero para conversación
conviene **BPE** con un tokenizador preentrenado:

```python
!pip install tiktoken
import tiktoken
enc = tiktoken.get_encoding("gpt2")
# Reemplaza las funciones encode/decode del notebook por:
# encode = enc.encode ; decode = enc.decode
# VOCAB_SIZE = enc.n_vocab  # ~50257
# Atención: n_embd y n_layer probablemente deban crecer (más vocab → más parámetros)
```

### 3. Aumentar el modelo
Con BPE y diálogo, recomiendo empezar con:
- `n_layer = 6`, `n_head = 6`, `n_embd = 384`
- `block_size = 256`
- `batch_size = 32`
- Más epochs (10-20) y lr scheduler más largo.

Eso da ~10M parámetros, entrenable en Colab free en ~30 min.

### 4. Formato de chat al inferir
En generación, mantén los últimos `block_size` tokens con el formato
`<user> ... <assistant> ... <user> ...` y corta la generación al primer
`<user>` que aparezca.

### 5. Siguientes saltos (si te enganchas)
- **PEFT / LoRA** para fine-tuning eficiente en modelos ya grandes (Qwen, LLaMA).
- **Datasets específicos**: PersonaChat, BlendedSkillTalk, EmpatheticDialogues.
- **Evaluación**: perplexity en held-out, BLEU/ROUGE, o evaluación humana de coherencia.

> Recuerda: el notebook actual es el **baseline**. Lo importante es que tengas
> todo el pipeline end-to-end funcionando antes de complicar nada.
